In [1]:
import numpy as np

In [ ]:
def value_iteration_matrix_form(P, R, gamma, theta=1e-6):
    """
    P.shape = (n_states, n_actions, n_states)
    R.shape = (n_states, n_actions, n_states)
    """
    
    n_states, n_actions, _ = P.shape

    V = np.zeros(n_states)

    while True:
        
        old_V = V.copy()

        #Q[s,a]
        Q = np.sum(P*(R+gamma*old_V), axis=2)

        #V[s]
        V = np.max(Q, axis=1)
        delta = np.max(np.abs(V-old_V))

        if delta < theta:
            break

    # optimal policy
    policy = np.argmax(Q, axis=1)

    return V, policy

In [6]:
def value_iteration_disctionary_form(P, gamma, theta=1e-6):

    states = list(P.keys())

    V = {s: 0.0 for s in states}

    while True:

        delta = 0

        for s in states:

            actions = P[s]

            # terminal state
            if len(actions) == 0:
                continue

            old_v = V[s]

            q_values = []

            for a in actions:

                q = 0

                for prob, next_s, reward in P[s][a]:

                    q += prob * (
                        reward +
                        gamma * V[next_s]
                    )

                q_values.append(q)

            V[s] = max(q_values)

            delta = max(
                delta,
                abs(old_v - V[s])
            )

        if delta < theta:
            break

    # extract policy
    policy = {}

    for s in states:

        actions = P[s]

        if len(actions) == 0:
            policy[s] = None
            continue

        best_action = None
        best_q = -float("inf")

        for a in actions:

            q = 0

            for prob, next_s, reward in P[s][a]:

                q += prob * (
                    reward +
                    gamma * V[next_s]
                )

            if q > best_q:
                best_q = q
                best_action = a

        policy[s] = best_action

    return V, policy